**Objectives**
<ul>
<li>Understand the distinction between state and memory.</li>
<li>
Utilize the ShortTermMemory class to manage session memory.</li>
<li>
Implement an Agent class that:
<ul>
<li>
Accepts a session_id for each interaction.</li>
<li>
Stores state in memory under the appropriate session.</li>
<li>
Retrieves session history to provide context for new queries.</li></ul>
</li>
<li>
Demonstrate the agent's ability to continue conversations across multiple interactions.</li></ul>

In [ ]:
!pip install langchain-openai langchain-core langgraph

In [ ]:
import os
from typing import TypedDict, List, Optional, Dict, Any
from dotenv import load_dotenv

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import (
    HumanMessage,
    SystemMessage,
    AIMessage,
    ToolMessage
)
from langchain_core.tools import tool

load_dotenv()
OPENAI_API_KEY = os.getenv("")

In [ ]:
# TOOL
@tool
def get_games() -> str:
    """Returns information about games."""

    games = [
        {"name": "Zelda", "score": 98},
        {"name": "Mario", "score": 95},
        {"name": "Cyberpunk", "score": 70},
    ]

    best_game = max(games, key=lambda x: x["score"])
    worst_game = min(games, key=lambda x: x["score"])

    return (
        f"Best game: {best_game['name']} ({best_game['score']})\n"
        f"Worst game: {worst_game['name']} ({worst_game['score']})"
    )



In [ ]:
class AgentState(TypedDict):
    user_query: str
    messages: List[Any]
    current_tool_calls: Optional[List[dict]]
    session_id: str


In [ ]:
#Short Term Memory
class ShortTermMemory:

    def __init__(self):
        self.store: Dict[str, List[Any]] = {}

    def get_messages(self, session_id: str):
        return self.store.get(session_id, [])

    def save_messages(self, session_id: str, messages: List[Any]):
        self.store[session_id] = messages


In [ ]:
class MemoryAgent:

    def __init__(
        self,
        model_name: str = "gpt-4o-mini",
        instructions: str = "You are a helpful AI assistant.",
        temperature: float = 0.7
    ):

        self.instructions = instructions

        self.memory = ShortTermMemory()

        self.llm = ChatOpenAI(
            model=model_name,
            temperature=temperature,
            api_key=os.getenv("OPENAI_API_KEY")
        )

        self.tools = [get_games]

        # Bind tools to model
        self.llm_with_tools = self.llm.bind_tools(self.tools)

        # Build graph
        self.workflow = self._create_graph()

        #Graph Creation
    def _create_graph(self):

        workflow = StateGraph(AgentState)

        workflow.add_node("prepare_messages", self._prepare_messages)
        workflow.add_node("llm_node", self._llm_node)
        workflow.add_node("tool_node", self._tool_node)

        workflow.set_entry_point("prepare_messages")

        workflow.add_edge("prepare_messages", "llm_node")

        workflow.add_conditional_edges(
            "llm_node",
            self._should_continue,
            {
                "tool_node": "tool_node",
                END: END
            }
        )

        workflow.add_edge("tool_node", "llm_node")

        return workflow.compile()
    def _prepare_messages(self, state: AgentState):

        previous_messages = self.memory.get_messages(
            state["session_id"]
        )

        messages = [
            SystemMessage(content=self.instructions),
            *previous_messages,
            HumanMessage(content=state["user_query"])
        ]

        return {
            "messages": messages
        }
    def _llm_node(self, state: AgentState):
        OPENAI_API_KEY=""
        response = self.llm_with_tools.invoke(state["messages"])

        messages = state["messages"] + [response]

        tool_calls = response.tool_calls if response.tool_calls else None

        return {
            "messages": messages,
            "current_tool_calls": tool_calls
        }

    def _tool_node(self, state: AgentState):

        messages = state["messages"]

        for tool_call in state["current_tool_calls"]:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_call_id = tool_call["id"]

            selected_tool = next(
                (t for t in self.tools if t.name == tool_name),
                None
            )

            if selected_tool:

                result = selected_tool.invoke(tool_args)

                messages.append(
                    ToolMessage(
                        content=str(result),
                        tool_call_id=tool_call_id
                    )
                )

        return {
            "messages": messages,
            "current_tool_calls": None
        }
    def _should_continue(self, state: AgentState):

        if state.get("current_tool_calls"):
            return "tool_node"

        return END

    def invoke(
        self,
        query: str,
        session_id: str = "default"
    ):

        initial_state = {
            "user_query": query,
            "messages": [],
            "current_tool_calls": None,
            "session_id": session_id
        }

        result = self.workflow.invoke(initial_state)

        # Save conversation memory
        self.memory.save_messages(
            session_id,
            result["messages"]
        )

        # Return final AI response
        last_message = result["messages"][-1]

        return last_message.content


In [ ]:
agent = MemoryAgent(
    instructions="You help users answer questions about games."
)

In [ ]:
response1 = agent.invoke(
    "What is the best game?",
    session_id="games"
)

print("\nFIRST RESPONSE:")
print(response1)